<a href="https://colab.research.google.com/github/apackowska/MLAlgorithms/blob/main/EEE_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# question 1
import numpy as np
import h5py
import matplotlib.pyplot as plt


# =========================
# Data loading
# =========================

def load_h5_data(filepath):
    with h5py.File(filepath, "r") as f:
        print("Keys in file:", list(f.keys()))
        trainims = np.array(f["trainims"])
        testims = np.array(f["testims"])
        trainlbls = np.array(f["trainlbls"])
        testlbls = np.array(f["testlbls"])
    return trainims, trainlbls, testims, testlbls


def preprocess_images(images):
    images = images.astype(np.float32)

    if images.max() > 1.0:
        images = images / 255.0

    X = images.reshape(images.shape[0], -1)
    return X


def preprocess_labels(labels):
    y = labels.reshape(-1, 1)
    unique_vals = np.unique(y)
    print("Unique labels before mapping:", unique_vals)

    if set(unique_vals.tolist()) == {0, 1}:
        y = 2 * y - 1
    elif set(unique_vals.tolist()) == {-1, 1}:
        pass
    else:
        raise ValueError(f"Unexpected label values: {unique_vals}")

    return y.astype(np.float32)


# =========================
# Neural network
# =========================

def tanh(x):
    return np.tanh(x)


def tanh_derivative_from_activation(a):
    return 1.0 - a**2


def initialize_parameters(input_dim, hidden_dim, output_dim=1, init_scale=0.05, seed=42):
    rng = np.random.default_rng(seed)

    W1 = rng.normal(0.0, init_scale, size=(input_dim, hidden_dim))
    b1 = np.zeros((1, hidden_dim))
    W2 = rng.normal(0.0, init_scale, size=(hidden_dim, output_dim))
    b2 = np.zeros((1, output_dim))

    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}


def forward_pass(X, params):
    Z1 = X @ params["W1"] + params["b1"]
    A1 = tanh(Z1)

    Z2 = A1 @ params["W2"] + params["b2"]
    A2 = tanh(Z2)

    cache = {"X": X, "A1": A1, "A2": A2}
    return A2, cache


def compute_mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


def predict_labels(y_pred):
    return np.where(y_pred >= 0, 1.0, -1.0)


def classification_error(y_true, y_pred):
    y_hat = predict_labels(y_pred)
    accuracy = np.mean(y_hat == y_true)
    return 1.0 - accuracy


def backward_pass(y_true, params, cache):
    X = cache["X"]
    A1 = cache["A1"]
    A2 = cache["A2"]
    m = X.shape[0]

    dA2 = 2.0 * (A2 - y_true) / m
    dZ2 = dA2 * tanh_derivative_from_activation(A2)

    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    dA1 = dZ2 @ params["W2"].T
    dZ1 = dA1 * tanh_derivative_from_activation(A1)

    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}


def update_parameters(params, grads, lr):
    params["W1"] -= lr * grads["dW1"]
    params["b1"] -= lr * grads["db1"]
    params["W2"] -= lr * grads["dW2"]
    params["b2"] -= lr * grads["db2"]


def evaluate_model(X, y, params):
    y_pred, _ = forward_pass(X, params)
    mse = compute_mse(y, y_pred)
    cls_err = classification_error(y, y_pred)
    return mse, cls_err


# =========================
# Training
# =========================

def create_mini_batches(X, y, batch_size, rng):
    n = X.shape[0]
    indices = rng.permutation(n)

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch_idx = indices[start:end]
        yield X[batch_idx], y[batch_idx]


def train_network(
    X_train, y_train, X_test, y_test,
    hidden_dim=32,
    lr=0.1,
    batch_size=32,
    epochs=60,
    init_scale=0.05,
    seed=42,
    verbose=False
):
    input_dim = X_train.shape[1]
    params = initialize_parameters(
        input_dim=input_dim,
        hidden_dim=hidden_dim,
        output_dim=1,
        init_scale=init_scale,
        seed=seed
    )

    rng = np.random.default_rng(seed)

    history = {
        "train_mse": [],
        "test_mse": [],
        "train_cls_err": [],
        "test_cls_err": []
    }

    for epoch in range(epochs):
        for X_batch, y_batch in create_mini_batches(X_train, y_train, batch_size, rng):
            y_pred_batch, cache = forward_pass(X_batch, params)
            grads = backward_pass(y_batch, params, cache)
            update_parameters(params, grads, lr)

        train_mse, train_cls_err = evaluate_model(X_train, y_train, params)
        test_mse, test_cls_err = evaluate_model(X_test, y_test, params)

        history["train_mse"].append(train_mse)
        history["test_mse"].append(test_mse)
        history["train_cls_err"].append(train_cls_err)
        history["test_cls_err"].append(test_cls_err)

        if verbose and (epoch % 10 == 0 or epoch == epochs - 1):
            print(
                f"Epoch {epoch+1:3d}/{epochs} | "
                f"Train MSE: {train_mse:.4f} | Test MSE: {test_mse:.4f} | "
                f"Train Cls Err: {train_cls_err:.4f} | Test Cls Err: {test_cls_err:.4f}"
            )

    return params, history

def compare_hidden_sizes(X_train, y_train, X_test, y_test, best_params):

    N_low = max(4, best_params["hidden_dim"] // 2)
    N_star = best_params["hidden_dim"]
    N_high = best_params["hidden_dim"] * 2

    configs = {
        "N_low": N_low,
        "N_star": N_star,
        "N_high": N_high
    }

    histories = {}

    for name, hidden_dim in configs.items():

        print(f"\nTraining network with hidden size {hidden_dim}")

        _, history = train_network(
            X_train, y_train,
            X_test, y_test,
            hidden_dim=hidden_dim,
            lr=best_params["lr"],
            batch_size=best_params["batch_size"],
            epochs=100,
            init_scale=best_params["init_scale"],
            verbose=False
        )

        histories[name] = history

    return histories

def plot_hidden_size_comparison(histories):

    epochs = len(histories["N_star"]["train_mse"])
    x = np.arange(1, epochs + 1)

    # MSE
    plt.figure(figsize=(10,5))

    for name, history in histories.items():
        plt.plot(x, history["test_mse"], label=f"{name} test")

    plt.xlabel("Epoch")
    plt.ylabel("Test MSE")
    plt.title("Test MSE vs Hidden Layer Size")
    plt.legend()
    plt.grid(True)
    plt.show()

    # classification error
    plt.figure(figsize=(10,5))

    for name, history in histories.items():
        plt.plot(x, history["test_cls_err"], label=f"{name} test")

    plt.xlabel("Epoch")
    plt.ylabel("Test Classification Error")
    plt.title("Classification Error vs Hidden Layer Size")
    plt.legend()
    plt.grid(True)
    plt.show()
# =========================
# Medium-size hyperparameter search
# =========================

def run_medium_experiments(X_train, y_train, X_test, y_test):
    hidden_dims = [16, 32, 64]
    lrs = [0.1, 0.3, 0.5]
    batch_sizes = [32, 64]
    init_scale = 0.05

    results = []

    exp_no = 0
    total = len(hidden_dims) * len(lrs) * len(batch_sizes)

    for hidden_dim in hidden_dims:
        for lr in lrs:
            for batch_size in batch_sizes:
                exp_no += 1
                print(
                    f"Experiment {exp_no}/{total}: "
                    f"hidden_dim={hidden_dim}, lr={lr}, batch_size={batch_size}"
                )

                _, history = train_network(
                    X_train, y_train, X_test, y_test,
                    hidden_dim=hidden_dim,
                    lr=lr,
                    batch_size=batch_size,
                    epochs=40,
                    init_scale=init_scale,
                    seed=42,
                    verbose=False
                )

                results.append({
                    "hidden_dim": hidden_dim,
                    "lr": lr,
                    "batch_size": batch_size,
                    "init_scale": init_scale,
                    "final_train_mse": history["train_mse"][-1],
                    "final_test_mse": history["test_mse"][-1],
                    "final_train_err": history["train_cls_err"][-1],
                    "final_test_err": history["test_cls_err"][-1]
                })

    results = sorted(results, key=lambda x: x["final_test_err"])
    return results


# =========================
# Plotting
# =========================

def plot_learning_curves(history):
    epochs = np.arange(1, len(history["train_mse"]) + 1)

    plt.figure(figsize=(10, 5))
    plt.plot(epochs, history["train_mse"], label="Training MSE")
    plt.plot(epochs, history["test_mse"], label="Testing MSE")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.title("Learning Curves: MSE")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(epochs, history["train_cls_err"], label="Training Classification Error")
    plt.plot(epochs, history["test_cls_err"], label="Testing Classification Error")
    plt.xlabel("Epoch")
    plt.ylabel("Classification Error")
    plt.title("Learning Curves: Classification Error")
    plt.legend()
    plt.grid(True)
    plt.show()

import numpy as np
import matplotlib.pyplot as plt


def initialize_parameters_2hidden(input_dim, hidden_dim1, hidden_dim2, output_dim=1, init_scale=0.05, seed=42):
    rng = np.random.default_rng(seed)

    W1 = rng.normal(0.0, init_scale, size=(input_dim, hidden_dim1))
    b1 = np.zeros((1, hidden_dim1))

    W2 = rng.normal(0.0, init_scale, size=(hidden_dim1, hidden_dim2))
    b2 = np.zeros((1, hidden_dim2))

    W3 = rng.normal(0.0, init_scale, size=(hidden_dim2, output_dim))
    b3 = np.zeros((1, output_dim))

    return {
        "W1": W1, "b1": b1,
        "W2": W2, "b2": b2,
        "W3": W3, "b3": b3
    }


def forward_pass_2hidden(X, params):
    Z1 = X @ params["W1"] + params["b1"]
    A1 = tanh(Z1)

    Z2 = A1 @ params["W2"] + params["b2"]
    A2 = tanh(Z2)

    Z3 = A2 @ params["W3"] + params["b3"]
    A3 = tanh(Z3)

    cache = {
        "X": X,
        "A1": A1,
        "A2": A2,
        "A3": A3
    }

    return A3, cache


def backward_pass_2hidden(y_true, params, cache):
    X = cache["X"]
    A1 = cache["A1"]
    A2 = cache["A2"]
    A3 = cache["A3"]

    m = X.shape[0]

    dA3 = 2.0 * (A3 - y_true) / m
    dZ3 = dA3 * tanh_derivative_from_activation(A3)

    dW3 = A2.T @ dZ3
    db3 = np.sum(dZ3, axis=0, keepdims=True)

    dA2 = dZ3 @ params["W3"].T
    dZ2 = dA2 * tanh_derivative_from_activation(A2)

    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    dA1 = dZ2 @ params["W2"].T
    dZ1 = dA1 * tanh_derivative_from_activation(A1)

    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    return {
        "dW1": dW1, "db1": db1,
        "dW2": dW2, "db2": db2,
        "dW3": dW3, "db3": db3
    }


def update_parameters_2hidden(params, grads, lr):
    params["W1"] -= lr * grads["dW1"]
    params["b1"] -= lr * grads["db1"]

    params["W2"] -= lr * grads["dW2"]
    params["b2"] -= lr * grads["db2"]

    params["W3"] -= lr * grads["dW3"]
    params["b3"] -= lr * grads["db3"]


def evaluate_model_2hidden(X, y, params):
    y_pred, _ = forward_pass_2hidden(X, params)
    mse = compute_mse(y, y_pred)
    cls_err = classification_error(y, y_pred)
    return mse, cls_err


def train_network_2hidden(
    X_train, y_train, X_test, y_test,
    hidden_dim1=32,
    hidden_dim2=16,
    lr=0.1,
    batch_size=32,
    epochs=100,
    init_scale=0.05,
    seed=42,
    verbose=False
):
    input_dim = X_train.shape[1]

    params = initialize_parameters_2hidden(
        input_dim=input_dim,
        hidden_dim1=hidden_dim1,
        hidden_dim2=hidden_dim2,
        output_dim=1,
        init_scale=init_scale,
        seed=seed
    )

    rng = np.random.default_rng(seed)

    history = {
        "train_mse": [],
        "test_mse": [],
        "train_cls_err": [],
        "test_cls_err": []
    }

    for epoch in range(epochs):
        for X_batch, y_batch in create_mini_batches(X_train, y_train, batch_size, rng):
            y_pred_batch, cache = forward_pass_2hidden(X_batch, params)
            grads = backward_pass_2hidden(y_batch, params, cache)
            update_parameters_2hidden(params, grads, lr)

        train_mse, train_cls_err = evaluate_model_2hidden(X_train, y_train, params)
        test_mse, test_cls_err = evaluate_model_2hidden(X_test, y_test, params)

        history["train_mse"].append(train_mse)
        history["test_mse"].append(test_mse)
        history["train_cls_err"].append(train_cls_err)
        history["test_cls_err"].append(test_cls_err)

        if verbose and (epoch % 10 == 0 or epoch == epochs - 1):
            print(
                f"Epoch {epoch+1:3d}/{epochs} | "
                f"Train MSE: {train_mse:.4f} | Test MSE: {test_mse:.4f} | "
                f"Train Cls Err: {train_cls_err:.4f} | Test Cls Err: {test_cls_err:.4f}"
            )

    return params, history


def run_2hidden_experiments(X_train, y_train, X_test, y_test):
    hidden_dim1_list = [32, 64]
    hidden_dim2_list = [16, 32]
    lrs = [0.1, 0.3, 0.5]
    batch_sizes = [32, 64]
    init_scale = 0.05

    results = []
    exp_no = 0
    total = len(hidden_dim1_list) * len(hidden_dim2_list) * len(lrs) * len(batch_sizes)

    for hidden_dim1 in hidden_dim1_list:
        for hidden_dim2 in hidden_dim2_list:
            for lr in lrs:
                for batch_size in batch_sizes:
                    exp_no += 1
                    print(
                        f"Experiment {exp_no}/{total}: "
                        f"h1={hidden_dim1}, h2={hidden_dim2}, lr={lr}, batch={batch_size}"
                    )

                    _, history = train_network_2hidden(
                        X_train, y_train, X_test, y_test,
                        hidden_dim1=hidden_dim1,
                        hidden_dim2=hidden_dim2,
                        lr=lr,
                        batch_size=batch_size,
                        epochs=40,
                        init_scale=init_scale,
                        seed=42,
                        verbose=False
                    )

                    results.append({
                        "hidden_dim1": hidden_dim1,
                        "hidden_dim2": hidden_dim2,
                        "lr": lr,
                        "batch_size": batch_size,
                        "init_scale": init_scale,
                        "final_train_mse": history["train_mse"][-1],
                        "final_test_mse": history["test_mse"][-1],
                        "final_train_err": history["train_cls_err"][-1],
                        "final_test_err": history["test_cls_err"][-1]
                    })

    results = sorted(results, key=lambda x: x["final_test_err"])
    return results


def plot_part_a_vs_d(history_a, history_d):
    epochs_a = np.arange(1, len(history_a["train_mse"]) + 1)
    epochs_d = np.arange(1, len(history_d["train_mse"]) + 1)

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_a, history_a["test_mse"], label="Part a: 1 hidden layer")
    plt.plot(epochs_d, history_d["test_mse"], label="Part d: 2 hidden layers")
    plt.xlabel("Epoch")
    plt.ylabel("Test MSE")
    plt.title("Part a vs Part d: Test MSE")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_a, history_a["test_cls_err"], label="Part a: 1 hidden layer")
    plt.plot(epochs_d, history_d["test_cls_err"], label="Part d: 2 hidden layers")
    plt.xlabel("Epoch")
    plt.ylabel("Test Classification Error")
    plt.title("Part a vs Part d: Test Classification Error")
    plt.legend()
    plt.grid(True)
    plt.show()


def initialize_velocities_2hidden(params):
    velocities = {}
    for key in params:
        velocities["v_" + key] = np.zeros_like(params[key])
    return velocities


def update_parameters_2hidden_momentum(params, grads, velocities, lr, alpha):
    velocities["v_W1"] = alpha * velocities["v_W1"] - lr * grads["dW1"]
    velocities["v_b1"] = alpha * velocities["v_b1"] - lr * grads["db1"]

    velocities["v_W2"] = alpha * velocities["v_W2"] - lr * grads["dW2"]
    velocities["v_b2"] = alpha * velocities["v_b2"] - lr * grads["db2"]

    velocities["v_W3"] = alpha * velocities["v_W3"] - lr * grads["dW3"]
    velocities["v_b3"] = alpha * velocities["v_b3"] - lr * grads["db3"]

    params["W1"] += velocities["v_W1"]
    params["b1"] += velocities["v_b1"]

    params["W2"] += velocities["v_W2"]
    params["b2"] += velocities["v_b2"]

    params["W3"] += velocities["v_W3"]
    params["b3"] += velocities["v_b3"]


def train_network_2hidden_momentum(
    X_train, y_train, X_test, y_test,
    hidden_dim1=32,
    hidden_dim2=16,
    lr=0.1,
    alpha=0.3,
    batch_size=32,
    epochs=100,
    init_scale=0.05,
    seed=42,
    verbose=False
):
    input_dim = X_train.shape[1]

    params = initialize_parameters_2hidden(
        input_dim=input_dim,
        hidden_dim1=hidden_dim1,
        hidden_dim2=hidden_dim2,
        output_dim=1,
        init_scale=init_scale,
        seed=seed
    )

    velocities = initialize_velocities_2hidden(params)
    rng = np.random.default_rng(seed)

    history = {
        "train_mse": [],
        "test_mse": [],
        "train_cls_err": [],
        "test_cls_err": []
    }

    for epoch in range(epochs):
        for X_batch, y_batch in create_mini_batches(X_train, y_train, batch_size, rng):
            y_pred_batch, cache = forward_pass_2hidden(X_batch, params)
            grads = backward_pass_2hidden(y_batch, params, cache)
            update_parameters_2hidden_momentum(params, grads, velocities, lr, alpha)

        train_mse, train_cls_err = evaluate_model_2hidden(X_train, y_train, params)
        test_mse, test_cls_err = evaluate_model_2hidden(X_test, y_test, params)

        history["train_mse"].append(train_mse)
        history["test_mse"].append(test_mse)
        history["train_cls_err"].append(train_cls_err)
        history["test_cls_err"].append(test_cls_err)

        if verbose and (epoch % 10 == 0 or epoch == epochs - 1):
            print(
                f"Epoch {epoch+1:3d}/{epochs} | "
                f"Train MSE: {train_mse:.4f} | Test MSE: {test_mse:.4f} | "
                f"Train Cls Err: {train_cls_err:.4f} | Test Cls Err: {test_cls_err:.4f}"
            )

    return params, history


def run_momentum_experiments(X_train, y_train, X_test, y_test, best_d):
    alphas = [0.1, 0.3, 0.5]
    results = []

    for alpha in alphas:
        print(f"Testing momentum alpha={alpha}")

        _, history = train_network_2hidden_momentum(
            X_train, y_train, X_test, y_test,
            hidden_dim1=best_d["hidden_dim1"],
            hidden_dim2=best_d["hidden_dim2"],
            lr=best_d["lr"],
            alpha=alpha,
            batch_size=best_d["batch_size"],
            epochs=40,
            init_scale=best_d["init_scale"],
            seed=42,
            verbose=False
        )

        results.append({
            "alpha": alpha,
            "final_train_mse": history["train_mse"][-1],
            "final_test_mse": history["test_mse"][-1],
            "final_train_err": history["train_cls_err"][-1],
            "final_test_err": history["test_cls_err"][-1]
        })

    results = sorted(results, key=lambda x: x["final_test_err"])
    return results


def plot_part_d_vs_e(history_d, history_e):
    epochs_d = np.arange(1, len(history_d["train_mse"]) + 1)
    epochs_e = np.arange(1, len(history_e["train_mse"]) + 1)

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_d, history_d["test_mse"], label="Part d: no momentum")
    plt.plot(epochs_e, history_e["test_mse"], label="Part e: with momentum")
    plt.xlabel("Epoch")
    plt.ylabel("Test MSE")
    plt.title("Part d vs Part e: Test MSE")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_d, history_d["test_cls_err"], label="Part d: no momentum")
    plt.plot(epochs_e, history_e["test_cls_err"], label="Part e: with momentum")
    plt.xlabel("Epoch")
    plt.ylabel("Test Classification Error")
    plt.title("Part d vs Part e: Test Classification Error")
    plt.legend()
    plt.grid(True)
    plt.show()

# =========================
# Main
# =========================

def task1_run():
    filepath = "assign2_data1.h5"

    trainims, trainlbls, testims, testlbls = load_h5_data(filepath)

    print("trainims shape:", trainims.shape)
    print("testims shape:", testims.shape)
    print("trainlbls shape:", trainlbls.shape)
    print("testlbls shape:", testlbls.shape)

    X_train = preprocess_images(trainims)
    X_test = preprocess_images(testims)
    y_train = preprocess_labels(trainlbls)
    y_test = preprocess_labels(testlbls)

    print("X_train shape:", X_train.shape)
    print("X_test shape:", X_test.shape)
    print("y_train shape:", y_train.shape)
    print("y_test shape:", y_test.shape)

    # Medium-size search
    results = run_medium_experiments(X_train, y_train, X_test, y_test)

    print("\nResults sorted by final test classification error:")
    for row in results:
        print(row)

    best = results[0]
    print("\nChosen parameters:")
    print(best)

    # Final longer training with selected parameters
    params, history = train_network(
        X_train, y_train, X_test, y_test,
        hidden_dim=best["hidden_dim"],
        lr=best["lr"],
        batch_size=best["batch_size"],
        epochs=100,
        init_scale=best["init_scale"],
        seed=42,
        verbose=True
    )

    plot_learning_curves(history)

    train_mse, train_err = evaluate_model(X_train, y_train, params)
    test_mse, test_err = evaluate_model(X_test, y_test, params)

    print("\nFinal results:")
    print(f"Training MSE: {train_mse:.4f}")
    print(f"Testing MSE: {test_mse:.4f}")
    print(f"Training classification error: {train_err:.4f}")
    print(f"Testing classification error: {test_err:.4f}")

    histories = compare_hidden_sizes(X_train, y_train, X_test, y_test, best)
    plot_hidden_size_comparison(histories)

    # ----- part d -----
    results_d = run_2hidden_experiments(X_train, y_train, X_test, y_test)

    print("\nPart d results sorted by final test classification error:")
    for row in results_d:
        print(row)

    best_d = results_d[0]
    print("\nChosen parameters for part d:")
    print(best_d)

    params_d, history_d = train_network_2hidden(
        X_train, y_train, X_test, y_test,
        hidden_dim1=best_d["hidden_dim1"],
        hidden_dim2=best_d["hidden_dim2"],
        lr=best_d["lr"],
        batch_size=best_d["batch_size"],
        epochs=100,
        init_scale=best_d["init_scale"],
        seed=42,
        verbose=True
    )

    plot_learning_curves(history_d)
    plot_part_a_vs_d(history, history_d)


    # ----- part e -----
    results_e = run_momentum_experiments(X_train, y_train, X_test, y_test, best_d)

    print("\nPart e results sorted by final test classification error:")
    for row in results_e:
        print(row)

    best_e = results_e[0]
    print("\nChosen momentum for part e:")
    print(best_e)

    params_e, history_e = train_network_2hidden_momentum(
        X_train, y_train, X_test, y_test,
        hidden_dim1=best_d["hidden_dim1"],
        hidden_dim2=best_d["hidden_dim2"],
        lr=best_d["lr"],
        alpha=best_e["alpha"],
        batch_size=best_d["batch_size"],
        epochs=100,
        init_scale=best_d["init_scale"],
        seed=42,
        verbose=True
    )

    plot_learning_curves(history_e)
    plot_part_d_vs_e(history_d, history_e)

In [2]:
import numpy as np
import h5py
import matplotlib.pyplot as plt


def task2_load_h5(filepath):
    with h5py.File(filepath, "r") as f:
        print("Keys in file:", list(f.keys()))
        data = {k: np.array(f[k]) for k in ["trainx", "traind", "valx", "vald", "testx", "testd"]}

    for k, v in data.items():
        print(f"{k} shape: {v.shape}")

    return data


def task2_prepare_inputs(x, vocab_size=250):
    x = np.array(x)

    if x.ndim != 2:
        raise ValueError(f"Expected 2D input array, got shape {x.shape}")

    if x.shape[1] == 3:
        X = x
    elif x.shape[0] == 3:
        X = x.T
    elif x.shape[1] == 3 * vocab_size:
        X = x.reshape(x.shape[0], 3, vocab_size).argmax(axis=2)
    elif x.shape[0] == 3 * vocab_size:
        X = x.T.reshape(x.shape[1], 3, vocab_size).argmax(axis=2)
    else:
        raise ValueError(
            f"Could not infer trigram layout from shape {x.shape}. "
            f"Expected (N,3), (3,N), (N,{3*vocab_size}) or ({3*vocab_size},N)."
        )

    X = X.astype(np.int64)

    if X.min() >= 1 and X.max() <= vocab_size:
        X = X - 1
    elif X.min() < 0 or X.max() >= vocab_size:
        raise ValueError(
            f"Input word indices out of range after parsing: min={X.min()}, max={X.max()}."
        )

    return X


def task2_prepare_targets(d, vocab_size=250):
    d = np.array(d)

    if d.ndim == 1:
        y = d
    elif d.ndim == 2 and 1 in d.shape:
        y = d.reshape(-1)
    elif d.ndim == 2 and d.shape[1] == vocab_size:
        y = d.argmax(axis=1)
    elif d.ndim == 2 and d.shape[0] == vocab_size:
        y = d.argmax(axis=0)
    else:
        raise ValueError(
            f"Could not infer target layout from shape {d.shape}. "
            f"Expected (N,), (N,1), (1,N), (N,{vocab_size}) or ({vocab_size},N)."
        )

    y = y.astype(np.int64)

    if y.min() >= 1 and y.max() <= vocab_size:
        y = y - 1
    elif y.min() < 0 or y.max() >= vocab_size:
        raise ValueError(
            f"Target indices out of range after parsing: min={y.min()}, max={y.max()}."
        )

    return y


def task2_sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def task2_softmax(z):
    z_shift = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z_shift)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


def task2_initialize_params(vocab_size, D, P, init_std=0.01, seed=42):
    rng = np.random.default_rng(seed)

    params = {
        "R":  rng.normal(0.0, init_std, size=(vocab_size, D)),
        "W1": rng.normal(0.0, init_std, size=(3 * D, P)),
        "b1": rng.normal(0.0, init_std, size=(1, P)),
        "W2": rng.normal(0.0, init_std, size=(P, vocab_size)),
        "b2": rng.normal(0.0, init_std, size=(1, vocab_size)),
    }

    return params


def task2_copy_params(params):
    return {k: v.copy() for k, v in params.items()}


def task2_initialize_velocities(params):
    return {k: np.zeros_like(v) for k, v in params.items()}


def task2_forward(X_idx, params):
    E = params["R"][X_idx]
    E_flat = E.reshape(X_idx.shape[0], -1)

    H_pre = E_flat @ params["W1"] + params["b1"]
    H = task2_sigmoid(H_pre)

    Z = H @ params["W2"] + params["b2"]
    probs = task2_softmax(Z)

    cache = {
        "X_idx": X_idx,
        "E": E,
        "E_flat": E_flat,
        "H": H,
        "probs": probs,
    }

    return probs, cache


def task2_cross_entropy_from_probs(probs, y):
    eps = 1e-12
    return -np.mean(np.log(probs[np.arange(len(y)), y] + eps))


def task2_accuracy_from_probs(probs, y):
    y_hat = np.argmax(probs, axis=1)
    return np.mean(y_hat == y)


def task2_backward(y, params, cache):
    X_idx = cache["X_idx"]
    E_flat = cache["E_flat"]
    H = cache["H"]
    probs = cache["probs"]

    batch_size = X_idx.shape[0]
    D = params["R"].shape[1]

    dZ = probs.copy()
    dZ[np.arange(batch_size), y] -= 1.0
    dZ /= batch_size

    dW2 = H.T @ dZ
    db2 = np.sum(dZ, axis=0, keepdims=True)

    dH = dZ @ params["W2"].T
    dH_pre = dH * H * (1.0 - H)

    dW1 = E_flat.T @ dH_pre
    db1 = np.sum(dH_pre, axis=0, keepdims=True)

    dE_flat = dH_pre @ params["W1"].T
    dE = dE_flat.reshape(batch_size, 3, D)

    dR = np.zeros_like(params["R"])
    for pos in range(3):
        np.add.at(dR, X_idx[:, pos], dE[:, pos, :])

    grads = {
        "R": dR,
        "W1": dW1,
        "b1": db1,
        "W2": dW2,
        "b2": db2,
    }

    return grads


def task2_update_params_momentum(params, grads, velocities, lr, alpha):
    for key in params:
        velocities[key] = alpha * velocities[key] - lr * grads[key]
        params[key] += velocities[key]


def task2_make_batches(X, y, batch_size, rng):
    n = X.shape[0]
    indices = rng.permutation(n)

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        idx = indices[start:end]
        yield X[idx], y[idx]


def task2_evaluate(X, y, params, eval_batch_size=4096):
    total_loss = 0.0
    total_correct = 0
    total_n = 0

    for start in range(0, X.shape[0], eval_batch_size):
        end = min(start + eval_batch_size, X.shape[0])
        xb = X[start:end]
        yb = y[start:end]

        probs, _ = task2_forward(xb, params)

        total_loss += -np.sum(np.log(probs[np.arange(len(yb)), yb] + 1e-12))
        total_correct += np.sum(np.argmax(probs, axis=1) == yb)
        total_n += len(yb)

    return total_loss / total_n, total_correct / total_n


def task2_train_model(
    X_train, y_train, X_val, y_val, X_test, y_test,
    D=32, P=256,
    vocab_size=250,
    lr=0.15,
    alpha=0.85,
    batch_size=200,
    max_epochs=30,
    init_std=0.01,
    patience=3,
    min_delta=1e-3,
    eval_every=5,
    seed=42,
    verbose=True,
):
    params = task2_initialize_params(vocab_size=vocab_size, D=D, P=P, init_std=init_std, seed=seed)
    velocities = task2_initialize_velocities(params)
    rng = np.random.default_rng(seed)

    history = {
        "epoch": [],
        "train_ce": [],
        "val_ce": [],
        "test_ce": [],
        "train_acc": [],
        "val_acc": [],
        "test_acc": [],
    }

    best_params = task2_copy_params(params)
    best_val_ce = np.inf
    best_epoch = 0
    wait = 0

    last_train_ce = np.nan
    last_train_acc = np.nan
    last_test_ce = np.nan
    last_test_acc = np.nan

    for epoch in range(1, max_epochs + 1):
        for xb, yb in task2_make_batches(X_train, y_train, batch_size, rng):
            probs, cache = task2_forward(xb, params)
            grads = task2_backward(yb, params, cache)
            task2_update_params_momentum(params, grads, velocities, lr, alpha)

        val_ce, val_acc = task2_evaluate(X_val, y_val, params)

        if epoch == 1 or epoch % eval_every == 0 or epoch == max_epochs:
            last_train_ce, last_train_acc = task2_evaluate(X_train, y_train, params)
            last_test_ce, last_test_acc = task2_evaluate(X_test, y_test, params)

        history["epoch"].append(epoch)
        history["train_ce"].append(last_train_ce)
        history["val_ce"].append(val_ce)
        history["test_ce"].append(last_test_ce)
        history["train_acc"].append(last_train_acc)
        history["val_acc"].append(val_acc)
        history["test_acc"].append(last_test_acc)

        if verbose:
            print(
                f"Epoch {epoch:02d}/{max_epochs} | "
                f"val CE={val_ce:.4f}, val acc={val_acc:.4f} | "
                f"train CE={last_train_ce:.4f}, test CE={last_test_ce:.4f}"
            )

        if val_ce < best_val_ce - min_delta:
            best_val_ce = val_ce
            best_epoch = epoch
            best_params = task2_copy_params(params)
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch}. Restoring best epoch {best_epoch}.")
                break

    params = best_params

    final_train_ce, final_train_acc = task2_evaluate(X_train, y_train, params)
    final_val_ce, final_val_acc = task2_evaluate(X_val, y_val, params)
    final_test_ce, final_test_acc = task2_evaluate(X_test, y_test, params)

    summary = {
        "best_epoch": best_epoch,
        "best_val_ce": best_val_ce,
        "final_train_ce": final_train_ce,
        "final_val_ce": final_val_ce,
        "final_test_ce": final_test_ce,
        "final_train_acc": final_train_acc,
        "final_val_acc": final_val_acc,
        "final_test_acc": final_test_acc,
    }

    return params, history, summary


def task2_plot_single_history(history, title_prefix="Task 2"):
    epochs = np.array(history["epoch"])

    plt.figure(figsize=(10, 5))
    plt.plot(epochs, history["train_ce"], label="Train CE")
    plt.plot(epochs, history["val_ce"], label="Validation CE")
    plt.plot(epochs, history["test_ce"], label="Test CE")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-Entropy")
    plt.title(f"{title_prefix}: Cross-Entropy")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(epochs, history["train_acc"], label="Train Accuracy")
    plt.plot(epochs, history["val_acc"], label="Validation Accuracy")
    plt.plot(epochs, history["test_acc"], label="Test Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{title_prefix}: Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()


def task2_plot_config_comparison(histories, metric_key="val_ce"):
    plt.figure(figsize=(10, 5))

    for label, history in histories.items():
        epochs = np.array(history["epoch"])
        plt.plot(epochs, history[metric_key], label=label)

    plt.xlabel("Epoch")
    ylabel = "Cross-Entropy" if "ce" in metric_key else "Accuracy"
    plt.ylabel(ylabel)
    plt.title(f"Task 2: {metric_key}")
    plt.legend()
    plt.grid(True)
    plt.show()


def task2_run_experiments(
    filepath="assign2_data2.h5",
    vocab_size=250,
    configs=((32, 256), (16, 128), (8, 64)),
    lr=0.15,
    alpha=0.85,
    batch_size=200,
    max_epochs=30,
    init_std=0.01,
    patience=5,
    eval_every=5,
    seed=42,
):
    data = task2_load_h5(filepath)

    X_train = task2_prepare_inputs(data["trainx"], vocab_size=vocab_size)
    y_train = task2_prepare_targets(data["traind"], vocab_size=vocab_size)

    X_val = task2_prepare_inputs(data["valx"], vocab_size=vocab_size)
    y_val = task2_prepare_targets(data["vald"], vocab_size=vocab_size)

    X_test = task2_prepare_inputs(data["testx"], vocab_size=vocab_size)
    y_test = task2_prepare_targets(data["testd"], vocab_size=vocab_size)

    print("Prepared shapes:")
    print("X_train:", X_train.shape, "y_train:", y_train.shape)
    print("X_val:", X_val.shape, "y_val:", y_val.shape)
    print("X_test:", X_test.shape, "y_test:", y_test.shape)

    results = []
    histories = {}

    for D, P in configs:
        label = f"D={D}, P={P}"
        print("\n" + "=" * 60)
        print(f"Training config {label}")
        print("=" * 60)

        _, history, summary = task2_train_model(
            X_train, y_train, X_val, y_val, X_test, y_test,
            D=D,
            P=P,
            vocab_size=vocab_size,
            lr=lr,
            alpha=alpha,
            batch_size=batch_size,
            max_epochs=max_epochs,
            init_std=init_std,
            patience=patience,
            eval_every=eval_every,
            seed=seed,
            verbose=True,
        )

        histories[label] = history
        results.append({
            "D": D,
            "P": P,
            **summary
        })

    results = sorted(results, key=lambda row: row["best_val_ce"])

    print("\nSorted results by best validation CE:")
    for row in results:
        print(row)

    best = results[0]
    best_label = f"D={best['D']}, P={best['P']}"

    task2_plot_config_comparison(histories, metric_key="val_ce")
    task2_plot_config_comparison(histories, metric_key="val_acc")
    task2_plot_single_history(histories[best_label], title_prefix=f"Best config ({best_label})")

    return results, histories, best

def task2_try_load_vocab(filepath):
    """
    Tries to find a vocabulary / word list inside the H5 file.
    Returns:
        vocab_list: list of strings of length 250, or None if not found
    """
    candidate_keys = ["words", "vocab", "vocabulary", "wordlist", "dictionary"]

    with h5py.File(filepath, "r") as f:
        keys = list(f.keys())

        for key in candidate_keys:
            if key in f:
                arr = np.array(f[key])

                # decode bytes if needed
                if arr.dtype.kind in {"S", "O", "U"}:
                    vocab = []
                    for x in arr.reshape(-1):
                        if isinstance(x, bytes):
                            vocab.append(x.decode("utf-8"))
                        else:
                            vocab.append(str(x))
                    return vocab

        # no obvious vocabulary found
        print("No vocabulary mapping found in H5 file. Predictions will be shown as word indices.")
        print("Available keys:", keys)

    return None


def task2_index_to_word(idx, vocab_list=None):
    if vocab_list is None:
        return f"word_{idx}"
    if 0 <= idx < len(vocab_list):
        return str(vocab_list[idx])
    return f"word_{idx}"


def task2_decode_trigram(trigram_idx, vocab_list=None):
    return [task2_index_to_word(int(i), vocab_list) for i in trigram_idx]


def task2_predict_probs(X_idx, params, batch_size=4096):
    """
    Returns full probability matrix for all examples in X_idx.
    Shape: (N, 250)
    """
    probs_all = []

    for start in range(0, X_idx.shape[0], batch_size):
        end = min(start + batch_size, X_idx.shape[0])
        probs, _ = task2_forward(X_idx[start:end], params)
        probs_all.append(probs)

    return np.vstack(probs_all)


def task2_topk_from_probs(probs_row, k=10):
    top_idx = np.argsort(probs_row)[::-1][:k]
    top_probs = probs_row[top_idx]
    return top_idx, top_probs


def task2_run_part_b(filepath, best_params, num_samples=5, top_k=10, seed=42):
    """
    Part b:
    - pick 5 sample trigrams from test set
    - generate predicted probabilities over all 250 words
    - list top 10 candidates for each trigram
    """
    data = task2_load_h5(filepath)

    X_test = task2_prepare_inputs(data["testx"], vocab_size=250)
    y_test = task2_prepare_targets(data["testd"], vocab_size=250)
    vocab_list = task2_try_load_vocab(filepath)

    rng = np.random.default_rng(seed)

    if X_test.shape[0] < num_samples:
        chosen_idx = np.arange(X_test.shape[0])
    else:
        chosen_idx = rng.choice(X_test.shape[0], size=num_samples, replace=False)

    chosen_idx = np.sort(chosen_idx)

    print("\n" + "=" * 80)
    print("TASK 2B: SAMPLE TEST TRIGRAM PREDICTIONS")
    print("=" * 80)

    for sample_no, idx in enumerate(chosen_idx, start=1):
        trigram = X_test[idx:idx+1]
        true_target = int(y_test[idx])

        probs, _ = task2_forward(trigram, best_params)
        probs_row = probs[0]  # shape (250,)

        top_idx, top_probs = task2_topk_from_probs(probs_row, k=top_k)

        trigram_words = task2_decode_trigram(trigram[0], vocab_list)
        true_word = task2_index_to_word(true_target, vocab_list)

        print(f"\nSample {sample_no}")
        print(f"Trigram: {trigram_words}")
        print(f"True 4th word: {true_word}")

        print("Top 10 predicted candidates:")
        for rank, (pred_idx, pred_prob) in enumerate(zip(top_idx, top_probs), start=1):
            pred_word = task2_index_to_word(int(pred_idx), vocab_list)
            print(f"  {rank:2d}. {pred_word:20s}  prob={pred_prob:.6f}")

        # optional quick sensibility cue
        if true_target in top_idx:
            true_rank = np.where(top_idx == true_target)[0][0] + 1
            print(f"True word appears in top {top_k}, at rank {true_rank}.")
        else:
            print(f"True word does not appear in top {top_k}.")

        # store full probability vector info if needed
        print(f"Stored full probability vector length: {len(probs_row)}")

def task2_run_experiments(
    filepath="assign2_data2.h5",
    vocab_size=250,
    configs=((32, 256), (16, 128), (8, 64)),
    lr=0.15,
    alpha=0.85,
    batch_size=200,
    max_epochs=30,
    init_std=0.01,
    patience=5,
    eval_every=5,
    seed=42,
):
    data = task2_load_h5(filepath)

    X_train = task2_prepare_inputs(data["trainx"], vocab_size=vocab_size)
    y_train = task2_prepare_targets(data["traind"], vocab_size=vocab_size)

    X_val = task2_prepare_inputs(data["valx"], vocab_size=vocab_size)
    y_val = task2_prepare_targets(data["vald"], vocab_size=vocab_size)

    X_test = task2_prepare_inputs(data["testx"], vocab_size=vocab_size)
    y_test = task2_prepare_targets(data["testd"], vocab_size=vocab_size)

    print("Prepared shapes:")
    print("X_train:", X_train.shape, "y_train:", y_train.shape)
    print("X_val:", X_val.shape, "y_val:", y_val.shape)
    print("X_test:", X_test.shape, "y_test:", y_test.shape)

    results = []
    histories = {}
    params_by_label = {}

    for D, P in configs:
        label = f"D={D}, P={P}"
        print("\n" + "=" * 60)
        print(f"Training config {label}")
        print("=" * 60)

        params, history, summary = task2_train_model(
            X_train, y_train, X_val, y_val, X_test, y_test,
            D=D,
            P=P,
            vocab_size=vocab_size,
            lr=lr,
            alpha=alpha,
            batch_size=batch_size,
            max_epochs=max_epochs,
            init_std=init_std,
            patience=patience,
            eval_every=eval_every,
            seed=seed,
            verbose=True,
        )

        histories[label] = history
        params_by_label[label] = params
        results.append({
            "D": D,
            "P": P,
            **summary
        })

    results = sorted(results, key=lambda row: row["best_val_ce"])

    print("\nSorted results by best validation CE:")
    for row in results:
        print(row)

    best = results[0]
    best_label = f"D={best['D']}, P={best['P']}"
    best_params = params_by_label[best_label]

    task2_plot_config_comparison(histories, metric_key="val_ce")
    task2_plot_config_comparison(histories, metric_key="val_acc")
    task2_plot_single_history(histories[best_label], title_prefix=f"Best config ({best_label})")

    return results, histories, best, best_params

def task2_run():
    results, histories, best, best_params = task2_run_experiments(filepath="assign2_data2.h5")

    task2_run_part_b(
        filepath="assign2_data2.h5",
        best_params=best_params,
        num_samples=5,
        top_k=10,
        seed=42
    )

    return results, histories, best, best_params

In [3]:
import argparse

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("task", choices=["task1", "task2"])
    # Fix: Explicitly pass the argument to parse_args() instead of letting it parse sys.argv
    args = parser.parse_args(['task2']) # Change to ['task2'] to run the second task

    if args.task == "task1":
        task1_run()
    elif args.task == "task2":
        task2_run()

Keys in file: ['testd', 'testx', 'traind', 'trainx', 'vald', 'valx', 'words']
trainx shape: (372500, 3)
traind shape: (372500,)
valx shape: (46500, 3)
vald shape: (46500,)
testx shape: (46500, 3)
testd shape: (46500,)
Prepared shapes:
X_train: (372500, 3) y_train: (372500,)
X_val: (46500, 3) y_val: (46500,)
X_test: (46500, 3) y_test: (46500,)

Training config D=32, P=256


KeyboardInterrupt: 